# 🚀 LlamaIndex - Query Engine সম্পূর্ণ গাইড

## Query Engine কী?
Query Engine হলো LlamaIndex-এর **হৃদপিণ্ড**।  
তুমি যখন ডকুমেন্ট থেকে index বানাস, তখন Query Engine সেই index-কে **প্রশ্ন করে উত্তর দেয়**।

## পুরো RAG Flow:
```
Documents → Splitter → Embedding → VectorStore → Index
                                                    ↓
Question → Query Engine → Retriever → LLM → Answer
```

## এই নোটবুকে যা শিখবে:
1. ✅ Basic Query Engine
2. ✅ Response দেখা (text, sources)
3. ✅ Query Engine Parameters (top_k, similarity)
4. ✅ Streaming Response
5. ✅ Custom Prompt দেওয়া
6. ✅ Retriever আলাদাভাবে ব্যবহার করা
7. ✅ Chat Engine (Memory সহ)
8. ✅ Index Persistence (Disk-এ Save/Load)

---
## ⚙️ Step 1: Setup — Model ও Documents তৈরি করা
প্রতিটা নোটবুকে এই setup আগে করতে হবে।

In [2]:
# প্রয়োজনীয় লাইব্রেরি import
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import MockLLM      # ফ্রিতে টেস্ট করার জন্য
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

# ─── Embedding Model সেট করা (HuggingFace - ফ্রি) ───
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # ছোট ও fast মডেল
)

# ─── LLM সেট করা ───
# Option A: MockLLM (ফ্রি, টেস্টের জন্য - real উত্তর দেবে না)
Settings.llm = MockLLM(max_tokens=256)

# Option B: OpenAI (real উত্তর দেবে, API key লাগবে)
# from dotenv import load_dotenv
# from llama_index.llms.openai import OpenAI
# load_dotenv()
# Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

print("✅ Settings সম্পন্ন!")

ModuleNotFoundError: No module named 'llama_index.embeddings.huggingface'

In [ ]:
# ─── Documents লোড করা ───
documents = SimpleDirectoryReader(
    input_dir="Data"
).load_data()

print(f"📄 মোট {len(documents)}টি ডকুমেন্ট লোড হয়েছে")
for doc in documents:
    print(f"  → {doc.metadata.get('file_name', 'Unknown')}")

In [ ]:
# ─── Index তৈরি করা ───
# SentenceSplitter দিয়ে ডকুমেন্ট ভাগ করব
splitter = SentenceSplitter(
    chunk_size=256,      # প্রতিটি chunk-এ সর্বোচ্চ 256 character
    chunk_overlap=50     # পরের chunk-এ 50 character আগের chunk থেকে নেবে
)

# VectorStoreIndex তৈরি
index = VectorStoreIndex.from_documents(
    documents=documents,
    transformations=[splitter]
)

print(f"✅ Index সফলভাবে তৈরি হয়েছে!")
print(f"   Type: {type(index)}")

---
## 📌 Step 2: Basic Query Engine
`as_query_engine()` দিয়ে সবচেয়ে সহজ উপায়ে প্রশ্ন করা।

In [1]:
# ─── Basic Query Engine তৈরি ───
query_engine = index.as_query_engine()

# প্রশ্ন করা
response = query_engine.query("What is LlamaIndex?")

# উত্তর দেখা
print("🤖 উত্তর:")
print(response)

NameError: name 'index' is not defined

---
## 📌 Step 3: Response অবজেক্ট বিস্তারিত দেখা
Response শুধু text না, এর ভেতরে আরও অনেক তথ্য আছে!

In [ ]:
# ─── Response-এর বিভিন্ন অংশ দেখা ───
response = query_engine.query("What is RAG?")

# 1. মূল উত্তর (text)
print("━" * 50)
print("📝 উত্তর (response.response):")
print(response.response)

print("\n" + "━" * 50)
# 2. Source Nodes — কোথা থেকে উত্তর নেওয়া হয়েছে
print("📂 Source Nodes (কোন ডকুমেন্ট থেকে নেওয়া হয়েছে):")
for i, node in enumerate(response.source_nodes):
    print(f"\n  🔹 Node {i+1}:")
    print(f"     Score (মিলের হার): {node.score:.4f}")
    print(f"     File: {node.metadata.get('file_name', 'N/A')}")
    print(f"     Text (প্রথম 150 char): {node.text[:150]}...")

---
## 📌 Step 4: Query Engine Parameters (কাস্টমাইজ করা)
`similarity_top_k` — কতটি chunk থেকে উত্তর খুঁজবে সেটা ঠিক করা।

In [ ]:
# ─── similarity_top_k সহ Query Engine ───
# similarity_top_k=1 মানে শুধু সবচেয়ে মিলে যাওয়া 1টি chunk থেকে উত্তর দেবে
# similarity_top_k=5 মানে সবচেয়ে ভালো 5টি chunk থেকে উত্তর দেবে

query_engine_top3 = index.as_query_engine(
    similarity_top_k=3   # ডিফল্ট হলো 2
)

response = query_engine_top3.query("What topics are covered?")

print("📝 উত্তর:")
print(response)

print(f"\n📊 মোট {len(response.source_nodes)}টি source node ব্যবহার হয়েছে")

In [ ]:
# ─── Response Mode পরিবর্তন করা ───
# response_mode ঠিক করে LLM কিভাবে উত্তর তৈরি করবে

# Mode 1: "compact" (ডিফল্ট) — chunks একসাথে করে উত্তর দেয়
qe_compact = index.as_query_engine(response_mode="compact")

# Mode 2: "tree_summarize" — বড় ডেটার জন্য, summary তৈরি করে
qe_tree = index.as_query_engine(response_mode="tree_summarize")

# Mode 3: "no_text" — শুধু source node দেখাবে, উত্তর দেবে না
qe_notext = index.as_query_engine(response_mode="no_text")

question = "What is LlamaIndex used for?"

print("=" * 50)
print("Mode: compact")
print(qe_compact.query(question))

print("\n" + "=" * 50)
print("Mode: tree_summarize")
print(qe_tree.query(question))

---
## 📌 Step 5: Streaming Response
বড় উত্তর আস্তে আস্তে দেখা (ChatGPT-এর মতো typing effect)।

In [ ]:
# ─── Streaming Query Engine ───
# streaming=True দিলে উত্তর আস্তে আস্তে আসবে

streaming_engine = index.as_query_engine(
    streaming=True,
    similarity_top_k=2
)

print("🔄 Streaming শুরু হচ্ছে...\n")

streaming_response = streaming_engine.query("Explain data ingestion in LlamaIndex")

# print_response_stream() দিয়ে stream করে দেখাও
streaming_response.print_response_stream()

print("\n\n✅ Streaming শেষ!")

---
## 📌 Step 6: Custom Prompt Template
LLM-কে নির্দিষ্ট ভাষায় বা নির্দিষ্ট ফরম্যাটে উত্তর দিতে বলা।

In [ ]:
from llama_index.core import PromptTemplate

# ─── Custom QA Prompt Template ───
# {context_str} = retrieved chunks (source documents)
# {query_str}   = user-এর প্রশ্ন

custom_prompt = PromptTemplate(
    """
    তুমি একজন সহায়ক AI assistant।
    নিচের context (তথ্য) ব্যবহার করে প্রশ্নের উত্তর দাও।
    যদি context-এ উত্তর না থাকে, বলো 'আমি জানি না'।
    
    Context:
    {context_str}
    
    প্রশ্ন: {query_str}
    
    উত্তর (বাংলায় দাও):
    """
)

# Custom prompt দিয়ে Query Engine তৈরি
custom_qe = index.as_query_engine(
    text_qa_template=custom_prompt,
    similarity_top_k=2
)

response = custom_qe.query("What is LlamaIndex?")
print("🤖 Custom Prompt দিয়ে উত্তর:")
print(response)

---
## 📌 Step 7: Retriever আলাদাভাবে ব্যবহার করা
Query Engine = **Retriever** + **Response Synthesizer**।
আলাদাভাবে Retriever ব্যবহার করলে শুধু relevant chunks পাবে, LLM call হবে না।

In [ ]:
# ─── Retriever আলাদাভাবে ব্যবহার ───
retriever = index.as_retriever(
    similarity_top_k=3   # সবচেয়ে কাছের 3টি chunk আনবে
)

# শুধু relevant chunks আনা (LLM call হবে না)
nodes = retriever.retrieve("What is RAG?")

print(f"📥 {len(nodes)}টি relevant chunk পাওয়া গেছে:\n")

for i, node in enumerate(nodes):
    print(f"{'━'*50}")
    print(f"🔹 Chunk {i+1}")
    print(f"   Similarity Score: {node.score:.4f}")
    print(f"   File: {node.metadata.get('file_name', 'N/A')}")
    print(f"   Text:\n   {node.text}")

In [ ]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterCondition

# ─── Metadata Filter দিয়ে Retriever ───
# শুধু নির্দিষ্ট ফাইল থেকে খুঁজবে

filters = MetadataFilters(
    filters=[
        MetadataFilter(
            key="file_name",
            value="practice_llamaindex.txt"
        )
    ]
)

filtered_retriever = index.as_retriever(
    similarity_top_k=3,
    filters=filters
)

nodes = filtered_retriever.retrieve("What is data loading?")

print("🔍 Filtered Retrieval (শুধু .txt ফাইল থেকে):")
for i, node in enumerate(nodes):
    print(f"\n  Node {i+1}: {node.metadata.get('file_name')}")
    print(f"  Score: {node.score:.4f}")
    print(f"  Text: {node.text[:100]}...")

---
## 📌 Step 8: Chat Engine — মেমরি সহ কথোপকথন
Chat Engine মনে রাখতে পারে আগের কথা।  
Query Engine একটা প্রশ্নের উত্তর দেয়, কিন্তু Chat Engine conversation চালিয়ে যেতে পারে।

In [ ]:
# ─── Chat Engine তৈরি ───
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",  # context মনে রেখে নতুন প্রশ্ন করে
    verbose=True                          # ভেতরে কী হচ্ছে দেখাবে
)

# প্রথম প্রশ্ন
print("👤 প্রশ্ন 1: What is LlamaIndex?")
response1 = chat_engine.chat("What is LlamaIndex?")
print(f"🤖 উত্তর: {response1}\n")

In [ ]:
# দ্বিতীয় প্রশ্ন — 'it' মানে কী? Chat Engine আগের context মনে রাখবে!
print("👤 প্রশ্ন 2: What are its key components?")
response2 = chat_engine.chat("What are its key components?")  # 'its' = LlamaIndex
print(f"🤖 উত্তর: {response2}\n")

# তৃতীয় প্রশ্ন
print("👤 প্রশ্ন 3: How does it help with RAG?")
response3 = chat_engine.chat("How does it help with RAG?")
print(f"🤖 উত্তর: {response3}")

In [ ]:
# ─── Chat History দেখা ───
print("📜 Chat History:")
for msg in chat_engine.chat_history:
    print(f"  [{msg.role.upper()}]: {msg.content[:100]}")

# Chat History মুছে ফেলা (নতুন করে শুরু করতে)
chat_engine.reset()
print("\n🔄 Chat history reset করা হয়েছে!")

---
## 📌 Step 9: Index Persistence — Disk-এ Save ও Load করা
প্রতিবার নতুন করে index বানানো সময়সাপেক্ষ।  
Index একবার তৈরি করে save করলে পরে শুধু load করলেই হবে!

In [ ]:
import os

# ─── Index Disk-এ Save করা ───
PERSIST_DIR = "./storage"   # এই ফোল্ডারে সেভ হবে

# Index সেভ করা
index.storage_context.persist(persist_dir=PERSIST_DIR)

print(f"✅ Index সফলভাবে '{PERSIST_DIR}' ফোল্ডারে save হয়েছে!")
print("📁 সেভ হওয়া ফাইলগুলো:")
for f in os.listdir(PERSIST_DIR):
    print(f"   → {f}")

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

# ─── Saved Index Load করা ───
# (Settings.embed_model আগে থেকে set থাকতে হবে)

PERSIST_DIR = "./storage"

# Disk থেকে storage context লোড
storage_context = StorageContext.from_defaults(
    persist_dir=PERSIST_DIR
)

# Index লোড করা
loaded_index = load_index_from_storage(storage_context)

print("✅ Index সফলভাবে load হয়েছে!")
print(f"   Type: {type(loaded_index)}")

# লোড করা index দিয়ে query করা
loaded_qe = loaded_index.as_query_engine()
response = loaded_qe.query("What is LlamaIndex?")
print(f"\n🤖 Loaded index থেকে উত্তর:\n{response}")

---
## 📌 Step 10: পুরো RAG Pipeline একসাথে (Production-ready)
এটাই Real-world project-এ ব্যবহার করা হয়।  
প্রথমবার → Index বানিয়ে save করবে  
পরের বার → save করা index load করবে (fast!)

In [ ]:
import os
from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.llms import MockLLM

# ════════════════════════════════════
#  ⚙️  Settings
# ════════════════════════════════════
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
Settings.llm = MockLLM(max_tokens=256)

PERSIST_DIR = "./storage"
DATA_DIR    = "Data"

# ════════════════════════════════════
#  📦  Index: Load বা Create
# ════════════════════════════════════
if os.path.exists(PERSIST_DIR):
    print("📂 আগের সেভ করা index পাওয়া গেছে। Load করা হচ্ছে...")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)
    print("✅ Index Load সম্পন্ন!")
else:
    print("🔨 নতুন Index তৈরি হচ্ছে...")
    documents = SimpleDirectoryReader(input_dir=DATA_DIR).load_data()
    splitter  = SentenceSplitter(chunk_size=256, chunk_overlap=50)
    index     = VectorStoreIndex.from_documents(
        documents=documents,
        transformations=[splitter]
    )
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"✅ Index তৈরি ও '{PERSIST_DIR}' ফোল্ডারে save সম্পন্ন!")

# ════════════════════════════════════
#  🔍  Query Engine
# ════════════════════════════════════
query_engine = index.as_query_engine(similarity_top_k=3)

# ════════════════════════════════════
#  ❓  প্রশ্ন করো
# ════════════════════════════════════
questions = [
    "What is LlamaIndex?",
    "What are the key components?",
    "How does RAG work?"
]

print("\n" + "=" * 55)
for q in questions:
    print(f"\n❓ প্রশ্ন: {q}")
    response = query_engine.query(q)
    print(f"🤖 উত্তর: {response.response}")
    print(f"📂 Sources: {[n.metadata.get('file_name','?') for n in response.source_nodes]}")
    print("-" * 55)

---
## 🎯 সারসংক্ষেপ — যা শিখলে

| বিষয় | কোড |
|-------|-----|
| Basic Query Engine | `index.as_query_engine()` |
| প্রশ্ন করা | `query_engine.query("প্রশ্ন")` |
| উত্তর দেখা | `response.response` |
| Source দেখা | `response.source_nodes` |
| Top-K সেট | `similarity_top_k=3` |
| Response Mode | `response_mode="compact"` |
| Streaming | `streaming=True` |
| Custom Prompt | `text_qa_template=PromptTemplate(...)` |
| Retriever আলাদা | `index.as_retriever()` |
| Filter করা | `MetadataFilters(...)` |
| Chat Engine | `index.as_chat_engine()` |
| Index Save | `index.storage_context.persist()` |
| Index Load | `load_index_from_storage(storage_context)` |

## পরবর্তী শেখার বিষয়
- 📌 **Evaluation** — RAG কতটা ভালো উত্তর দিচ্ছে পরিমাপ করা
- 📌 **Agents** — LlamaIndex দিয়ে AI Agent তৈরি করা
- 📌 **Sub-Question Query Engine** — জটিল প্রশ্ন ভেঙে উত্তর দেওয়া